In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 2000

candidate_id = [f"CAND-{str(i).zfill(4)}" for i in range(1, n+1)]

years_experience = np.random.choice(
    [1,2,3,4,5,6,7,8,10,12,15,18,20], size=n,
    p=[0.08,0.08,0.09,0.09,0.1,0.09,0.09,0.08,0.08,0.07,0.06,0.05,0.04]
)

current_level = np.where(
    years_experience <= 3, 'Junior',
    np.where(years_experience <= 7, 'Mid',
    np.where(years_experience <= 12, 'Senior',
    np.where(years_experience <= 17, 'Lead', 'Director')))
)

level_map = {'Junior':1,'Mid':2,'Senior':3,'Lead':4,'Director':5}
current_level_num = np.array([level_map[l] for l in current_level])

seniority_jump = np.random.choice([-1,0,1,2], n, p=[0.05,0.40,0.45,0.10])
offered_level_num = np.clip(current_level_num + seniority_jump, 1, 5)
level_reverse = {1:'Junior',2:'Mid',3:'Senior',4:'Lead',5:'Director'}
offered_level = np.array([level_reverse[l] for l in offered_level_num])

months_in_current_role = np.array([
    int(np.clip(np.random.normal(
        loc=18 + (level_map[lvl] * 6), scale=10
    ), 3, 84))
    for lvl in current_level
])

notice_period_days = np.random.choice(
    [0,14,30,60,90], n, p=[0.05,0.15,0.45,0.25,0.10]
)

salary_map = {
    'Junior':   (380000, 480000),
    'Mid':      (480000, 640000),
    'Senior':   (640000, 860000),
    'Lead':     (820000, 1050000),
    'Director': (980000, 1400000)
}

current_salary = np.array([
    np.random.randint(*salary_map[lvl]) for lvl in current_level
])

salary_lift_pct = np.round(np.random.normal(loc=14, scale=9, size=n), 1)
salary_lift_pct = np.clip(salary_lift_pct, -8, 45)
offered_salary = np.round(
    current_salary * (1 + salary_lift_pct / 100), -3
).astype(int)

process_length_days = np.random.choice(
    range(10,91), n,
    p=np.exp(-0.03*np.arange(10,91)) / np.exp(-0.03*np.arange(10,91)).sum()
)

response_time_hours = np.round(
    np.random.lognormal(mean=2.5, sigma=1.0, size=n), 1
)
response_time_hours = np.clip(response_time_hours, 0.5, 120)

interview_score = np.round(np.random.normal(loc=3.5, scale=0.7, size=n), 2)
interview_score = np.clip(interview_score, 1.0, 5.0)

recruiter_sentiment = np.round(np.random.normal(loc=3.6, scale=0.8, size=n), 1)
recruiter_sentiment = np.clip(recruiter_sentiment, 1.0, 5.0)

hiring_manager_sentiment = np.round(np.random.normal(loc=3.4, scale=0.9, size=n), 1)
hiring_manager_sentiment = np.clip(hiring_manager_sentiment, 1.0, 5.0)

candidate_nps = np.random.choice(
    range(-100,101), n,
    p=np.concatenate([
        np.linspace(0.001,0.003,100),
        np.linspace(0.003,0.012,101)
    ]) / np.concatenate([
        np.linspace(0.001,0.003,100),
        np.linspace(0.003,0.012,101)
    ]).sum()
)

competing_offers = np.random.choice([0,1], n, p=[0.58,0.42])

employer_brand_score = np.round(np.random.normal(loc=3.7, scale=0.6, size=n), 1)
employer_brand_score = np.clip(employer_brand_score, 1.0, 5.0)

remote_policy_match = np.random.choice([1,0], n, p=[0.62,0.38])
relocation_required = np.random.choice([0,1], n, p=[0.78,0.22])
visa_required       = np.random.choice([0,1], n, p=[0.85,0.15])

has_tech_test = np.random.choice([0,1], n, p=[0.35,0.65])
technical_test_score = np.where(
    has_tech_test == 1,
    np.round(np.random.normal(loc=68, scale=15, size=n), 1),
    np.nan
)
technical_test_score = np.where(technical_test_score > 100, 100, technical_test_score)

stage_drop_risk = np.round(
    (response_time_hours/120)*0.3 +
    (1 - recruiter_sentiment/5)*0.3 +
    (process_length_days/90)*0.2 +
    (competing_offers*0.2),
    3
)

def simulate_acceptance(i):
    score = 0.0
    if salary_lift_pct[i] > 20:   score += 3.5
    elif salary_lift_pct[i] > 10: score += 1.5
    elif salary_lift_pct[i] < 0:  score -= 3.0
    if seniority_jump[i] == 2:    score += 2.5
    elif seniority_jump[i] == 1:  score += 1.5
    elif seniority_jump[i] == -1: score -= 2.0
    if months_in_current_role[i] < 12:   score -= 1.5
    elif months_in_current_role[i] > 36: score += 1.2
    if competing_offers[i] == 1:  score -= 2.5
    if process_length_days[i] < 21:   score += 1.5
    elif process_length_days[i] > 55: score -= 1.5
    score += (hiring_manager_sentiment[i] - 3) * 1.2
    score += (recruiter_sentiment[i] - 3) * 0.8
    score += (candidate_nps[i] / 100) * 1.5
    if relocation_required[i] == 1: score -= 1.2
    if visa_required[i] == 1:       score -= 0.8
    if remote_policy_match[i] == 1: score += 1.0
    score += (employer_brand_score[i] - 3) * 0.7
    if response_time_hours[i] < 4:    score += 0.8
    elif response_time_hours[i] > 48: score -= 0.8
    prob = 1 / (1 + np.exp(-score * 0.28))
    return np.random.binomial(1, prob)

offer_accepted = np.array([simulate_acceptance(i) for i in range(n)])

df = pd.DataFrame({
    'candidate_id':             candidate_id,
    'current_level':            current_level,
    'offered_level':            offered_level,
    'seniority_jump':           seniority_jump,
    'years_experience':         years_experience,
    'months_in_current_role':   months_in_current_role,
    'notice_period_days':       notice_period_days,
    'current_salary':           current_salary,
    'offered_salary':           offered_salary,
    'salary_increase_pct':      salary_lift_pct,
    'remote_policy_match':      remote_policy_match,
    'interview_score':          interview_score,
    'recruiter_sentiment':      recruiter_sentiment,
    'hiring_manager_sentiment': hiring_manager_sentiment,
    'process_length_days':      process_length_days,
    'response_time_hours':      response_time_hours,
    'competing_offers':         competing_offers,
    'employer_brand_score':     employer_brand_score,
    'technical_test_score':     technical_test_score,
    'candidate_nps':            candidate_nps,
    'relocation_required':      relocation_required,
    'visa_required':            visa_required,
    'stage_drop_risk':          stage_drop_risk,
    'offer_accepted':           offer_accepted
})

print(f"✅ Done — {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Acceptance rate: {df['offer_accepted'].mean():.1%}")
print(df.head())

✅ Done — 2000 rows, 24 columns
Acceptance rate: 73.7%
  candidate_id current_level offered_level  seniority_jump  years_experience  \
0    CAND-0001           Mid           Mid               0                 5   
1    CAND-0002      Director      Director               0                18   
2    CAND-0003        Senior      Director               2                10   
3    CAND-0004           Mid           Mid               0                 7   
4    CAND-0005        Junior        Junior               0                 2   

   months_in_current_role  notice_period_days  current_salary  offered_salary  \
0                      43                  14          483823          580000   
1                      51                  30         1124526         1255000   
2                      29                  30          775004          899000   
3                      34                  30          621292          662000   
4                      19                  30          45409

In [5]:
df.to_csv('synthetic_candidates.csv', index=False)
print("✅ File saved — download it from the Files panel on the left")

✅ File saved — download it from the Files panel on the left
